# 🛰️ NN_segmentation — Kaggle Launcher

---

## 📋 Setup iniziale

Prima di eseguire, collega il dataset dal pannello **"+ Add Data"** (barra laterale destra):

| Dataset Kaggle da allegare | Come trovarlo |
|---|---|
| **Qualsiasi dataset in formato images/label** | Cerca: `global-land-cover-mapping-openearthmap` (o altri) |

> ✅ **Non devi caricare niente dal tuo PC!**  
> Il dataset OpenEarthMap è già disponibile pubblicamente su Kaggle.  


## ① Clone / Pull del codice da GitHub

In [ ]:
import os

REPO_URL  = 'https://github.com/robertopassante/NN_segmentation.git'
REPO_NAME = 'NN_segmentation'
REPO_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print(f'[INFO] Clonazione {REPO_NAME}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'[INFO] Repo già presente. Aggiornamento...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'[OK] Working directory: {os.getcwd()}')

## ② Installazione dipendenze e pesi pre-addestrati RSP Swin-T

In [ ]:
import os

!pip install -q -r requirements.txt
!pip install -q yacs safetensors rasterio albumentations

# Pesi RSP Swin-T (download solo se assenti)
WEIGHTS_PATH = '/kaggle/working/NN_segmentation/rsp-swin-t-ckpt.safetensors'
if not os.path.exists(WEIGHTS_PATH):
    print('[INFO] Download pesi RSP Swin-T...')
    !wget -q -O {WEIGHTS_PATH} \
        https://huggingface.co/BiliSakura/RSP-Swin-T/resolve/main/model.safetensors
    print('[OK] Pesi scaricati.')
else:
    print('[SKIP] Pesi RSP già presenti.')

## ③ Verifica dataset allegato

> Il sistema rileva automaticamente qualsiasi dataset strutturato in `images/train`.


In [ ]:
import os
import glob

# ── Rilevamento automatico della cartella del dataset ────────────────────────
# Troviamo automaticamente il path cercando la cartella 'images/train'
found_paths = glob.glob('/kaggle/input/**/images/train', recursive=True)
if found_paths:
    # Prende la cartella "padre" di images/train
    INPUT_DIR = os.path.dirname(os.path.dirname(found_paths[0]))
else:
    INPUT_DIR = '/kaggle/input/global-land-cover-mapping-openearthmap'

# Verifica che il dataset sia allegato
if not os.path.exists(INPUT_DIR) or not os.path.exists(os.path.join(INPUT_DIR, 'images')):
    print(f'❌ Dataset non trovato (path esplorato: {INPUT_DIR})')
    print(f'   → Vai su "+ Add Data" e cerca "global-land-cover-mapping-openearthmap"')
    raise FileNotFoundError(f'Dataset mancante: {INPUT_DIR}')

# Conta le immagini disponibili
for split in ['train', 'val']:
    img_dir = os.path.join(INPUT_DIR, 'images', split)
    lbl_dir = os.path.join(INPUT_DIR, 'label',  split)
    n_img = len([f for f in os.listdir(img_dir) if f.endswith('.tif')]) if os.path.exists(img_dir) else 0
    n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.tif')]) if os.path.exists(lbl_dir) else 0
    print(f'[{split}] Immagini: {n_img} | Mask: {n_lbl}')

print(f'\n✅ Dataset intercettato correttamente in: {INPUT_DIR}')

## ④ Training (Full Dataset)
L'addestramento userà tutte le immagini rilevate.

In [ ]:
%cd /kaggle/working/NN_segmentation
!python main_kaggle.py

## ⑥ Fase 5: Generazione Pseudo-Labels (Advanced Task)
Per sbloccare il potenziale del modello (80%+ mIoU), useremo il modello appena addestrato in combinazione con SAM di Meta per auto-etichettare enormi quantità di nuove immagini.\n
**Nota:** Cambia `UNLABELED_DIR` con il path di una cartella contenente immagini senza maschere.

In [ ]:
import os
import glob
%cd /kaggle/working/NN_segmentation

# 1. Trova dinamicamente la cartella delle immagini di test senza etichetta
found_paths = glob.glob('/kaggle/input/**/images/test', recursive=True)
if found_paths:
    UNLABELED_DIR = found_paths[0]
else:
    UNLABELED_DIR = '/kaggle/input/global-land-cover-mapping-openearthmap/images/test'

OUT_PSEUDO_DIR = '/kaggle/working/pseudo_dataset'

# 2. Download pesi SAM se non presenti
SAM_WEIGHTS = '/kaggle/working/NN_segmentation/sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_WEIGHTS):
    print('[INFO] Scarico Segment Anything Model (SAM) weights...')
    !wget -q -O {SAM_WEIGHTS} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

# 3. Esegui il processo massivo
!python pseudo_labeling.py \
    --input_dir {UNLABELED_DIR} \
    --output_dir {OUT_PSEUDO_DIR} \
    --unet_ckpt /kaggle/input/datasets/robertopassante/pretrained-satellite-model/best_model.pth \
    --sam_ckpt {SAM_WEIGHTS}

print('\n✅ Pseudo-Labels generate con successo nella cartella:', OUT_PSEUDO_DIR)


## ⑥ Fase 6: Fusione Dataset (Originale + Pseudo-Labels)
Uniamo il dataset di partenza con le nuove maschere per creare un dataset combinato ed evitare il Catastrophic Forgetting.

In [ ]:
import os
import glob
import shutil
from tqdm import tqdm

# 1. Trova dinamicamente il dataset originale su Kaggle
found_paths = glob.glob('/kaggle/input/**/images/train', recursive=True)
if found_paths:
    ORIGINAL_DIR = os.path.dirname(os.path.dirname(found_paths[0]))
else:
    ORIGINAL_DIR = '/kaggle/input/global-land-cover-mapping-openearthmap'

COMBINED_DIR = '/kaggle/working/combined_dataset'
PSEUDO_DIR = '/kaggle/working/pseudo_dataset'

print(f"[INFO] Creazione dataset combinato in: {COMBINED_DIR}")
for split in ['train', 'val']:
    os.makedirs(os.path.join(COMBINED_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(COMBINED_DIR, 'label', split), exist_ok=True)

def copy_files_with_progress(src_folder, dst_folder, desc):
    if not os.path.exists(src_folder):
        return
    files = os.listdir(src_folder)
    for f in tqdm(files, desc=desc, leave=False):
        shutil.copy2(os.path.join(src_folder, f), os.path.join(dst_folder, f))

# 2. Copia i dati originali
print(f"[INFO] Copia dei dati ORIGINALI da {ORIGINAL_DIR}...")
for split in ['train', 'val']:
    copy_files_with_progress(os.path.join(ORIGINAL_DIR, 'images', split), os.path.join(COMBINED_DIR, 'images', split), f'Originali Images {split}')
    copy_files_with_progress(os.path.join(ORIGINAL_DIR, 'label', split), os.path.join(COMBINED_DIR, 'label', split), f'Originali Label {split}')

# 3. Copia le pseudo-labels
print(f"[INFO] Copia delle PSEUDO-LABELS da {PSEUDO_DIR}...")
copy_files_with_progress(os.path.join(PSEUDO_DIR, 'images'), os.path.join(COMBINED_DIR, 'images', 'train'), 'Pseudo Images')
if os.path.exists(os.path.join(PSEUDO_DIR, 'labels')):
    copy_files_with_progress(os.path.join(PSEUDO_DIR, 'labels'), os.path.join(COMBINED_DIR, 'label', 'train'), 'Pseudo Labels')
else:
    copy_files_with_progress(os.path.join(PSEUDO_DIR, 'label'), os.path.join(COMBINED_DIR, 'label', 'train'), 'Pseudo Labels')

# 4. Check finale
n_train_img = len(os.listdir(os.path.join(COMBINED_DIR, 'images', 'train')))
print(f"\n✅ FUSIONE COMPLETATA! Immagini totali di training: {n_train_img}")


## ⑦ Fase 7: Training Avanzato (Fine-Tuning)
Facciamo ripartire il training usando il nuovo dataset combinato e caricando i pesi pre-addestrati del modello precedente.

In [ ]:
import os
%cd /kaggle/working/NN_segmentation

print("Inizio del Secondo Training (Fine-Tuning su dataset combinato)...")

# MODIFICA IMPORTANTE:
# Se hai caricato best_model.pth come Kaggle Dataset (per saltare il primo training),
# modifica "--resume_from" facendolo puntare a quel path, ad esempio:
# --resume_from /kaggle/input/my-pretrained-satellite-models/best_model.pth

!python main_kaggle.py \
  --data_dir /kaggle/working/combined_dataset \
  --resume_from /kaggle/input/datasets/robertopassante/pretrained-satellite-model/best_model.pth
